# Importing Libraries

In [67]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import shutil
import os

# Reading Datasets

In [68]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")  
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
product_category = pd.read_csv("../data/raw/product_category_name_translation.csv")

# Data Cleaning

## Missing Values

In [69]:
customers.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [70]:
geolocation.isna().sum()    

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [71]:
order_reviews.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [72]:
order_items.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [73]:
order_payments.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [74]:
order_reviews.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [75]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [76]:
products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [77]:
sellers.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [78]:
product_category.isna().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

## Duplicate Values

In [79]:
customers.duplicated().sum()

np.int64(0)

In [80]:
geolocation.duplicated().sum()  

np.int64(261831)

In [ ]:
# Verificando se o mesmo CEP possui diferentes cidades ou estados
geolocation.groupby('geolocation_zip_code_prefix')['geolocation_city'].nunique().sort_values(ascending=False).head(10)

geolocation_zip_code_prefix
17970    5
13455    5
78290    5
6900     5
13318    5
13457    5
13454    5
28950    5
42850    5
76958    4
Name: geolocation_city, dtype: int64

In [113]:
# Inconsistencia encontrada: mesmo CEP mapeado para multiplas cidades (ate 5).
# Estrategia: mantido a cidade/estado mais frequente por CEP e calculado a media das coordenadas.

In [ ]:
# Mantendo a cidade/estado mais frequente por CEP
geolocation = geolocation.groupby('geolocation_zip_code_prefix').agg(
    geolocation_lat=('geolocation_lat', 'mean'),
    geolocation_lng=('geolocation_lng', 'mean'),
    geolocation_city=('geolocation_city', lambda x: x.value_counts().index[0]),
    geolocation_state=('geolocation_state', lambda x: x.value_counts().index[0])
).reset_index()

In [84]:
order_items.duplicated().sum()

np.int64(0)

In [85]:
order_payments.duplicated().sum()

np.int64(0)

In [86]:
order_reviews.duplicated().sum()

np.int64(0)

In [87]:
orders.duplicated().sum()

np.int64(0)

In [88]:
products.duplicated().sum()

np.int64(0)

In [89]:
sellers.duplicated().sum()

np.int64(0)

In [90]:
product_category.duplicated().sum()

np.int64(0)

## Data Types Conversion

In [ ]:
# Convertendo coluna de str para datetime
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

In [ ]:
# Convertendo coluna de str para datetime
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

## Standardization

In [ ]:
# Padronizando customers
customers['customer_city'] = customers['customer_city'].str.strip().str.lower().str.replace(r'\s+', ' ', regex=True) # removes spaces, converts to lowercase and removes double spaces
customers['customer_state'] = customers['customer_state'].str.strip().str.upper() # removes spaces and converts to uppercase

In [ ]:
# Padronizando  geolocation
geolocation['geolocation_city'] = geolocation['geolocation_city'].str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)
geolocation['geolocation_state'] = geolocation['geolocation_state'].str.strip().str.upper()

In [ ]:
# Padronizando  order_payments
order_payments['payment_type'] = order_payments['payment_type'].str.strip().str.lower()

In [ ]:
# Padronizando  order_reviews
order_reviews['review_comment_title'] = order_reviews['review_comment_title'].str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)
order_reviews['review_comment_message'] = order_reviews['review_comment_message'].str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)


In [ ]:
# Padronizando  orders
orders['order_status'] = orders['order_status'].str.strip().str.lower()

In [ ]:
# Padronizando  products
products['product_category_name'] = products['product_category_name'].str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)

In [ ]:
# Padronizando sellers
sellers['seller_city'] = sellers['seller_city'].str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)
sellers['seller_state'] = sellers['seller_state'].str.strip().str.upper()

In [ ]:
# Padronizando  product_category
product_category['product_category_name'] = product_category['product_category_name'].str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)
product_category['product_category_name_english'] = product_category['product_category_name_english'].str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)

## Outliers

In [ ]:
# order_items

cols = ['price', 'freight_value']

for col in cols:
    Q1 = order_items[col].quantile(0.25)  # primeiro quartil
    Q3 = order_items[col].quantile(0.75)  # terceiro quartil
    IQR = Q3 - Q1                         # intervalo interquartil
    lower = Q1 - 1.5 * IQR                # limite inferior
    upper = Q3 + 1.5 * IQR                # limite superior
    outliers = order_items[(order_items[col] < lower) | (order_items[col] > upper)]

    print(f"\n{col}")
    print(f"   Lower bound : {lower:.2f}")
    print(f"   Upper bound : {upper:.2f}")
    print(f"   Outliers    : {len(outliers)} ({(len(outliers)/len(order_items)*100):.2f}%)")
    print(f"   Max value   : {order_items[col].max():.2f}")
    print(f"   Min value   : {order_items[col].min():.2f}")


price
   Lower bound : -102.60
   Upper bound : 277.40
   Outliers    : 8427 (7.48%)
   Max value   : 6735.00
   Min value   : 0.85

freight_value
   Lower bound : 0.98
   Upper bound : 33.25
   Outliers    : 12134 (10.77%)
   Max value   : 409.68
   Min value   : 0.00


In [114]:
# order_payments 

cols = ['payment_installments', 'payment_value']

for col in cols:
    Q1 = order_payments[col].quantile(0.25)
    Q3 = order_payments[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = order_payments[(order_payments[col] < lower) | (order_payments[col] > upper)]

    print(f"\n{col}")
    print(f"   Limite inferior : {lower:.2f}")
    print(f"   Limite superior : {upper:.2f}")
    print(f"   Outliers        : {len(outliers)} ({(len(outliers)/len(order_payments)*100):.2f}%)")
    print(f"   Valor maximo    : {order_payments[col].max():.2f}")
    print(f"   Valor minimo    : {order_payments[col].min():.2f}")


payment_installments
   Limite inferior : -3.50
   Limite superior : 8.50
   Outliers        : 6313 (6.08%)
   Valor maximo    : 24.00
   Valor minimo    : 1.00

payment_value
   Limite inferior : -115.79
   Limite superior : 344.42
   Outliers        : 7981 (7.68%)
   Valor maximo    : 13664.08
   Valor minimo    : 0.00


In [115]:
# products 

cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

for col in cols:
    Q1 = products[col].quantile(0.25)
    Q3 = products[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = products[(products[col] < lower) | (products[col] > upper)]

    print(f"\n{col}")
    print(f"   Limite inferior : {lower:.2f}")
    print(f"   Limite superior : {upper:.2f}")
    print(f"   Outliers        : {len(outliers)} ({(len(outliers)/len(products)*100):.2f}%)")
    print(f"   Valor maximo    : {products[col].max():.2f}")
    print(f"   Valor minimo    : {products[col].min():.2f}")


product_weight_g
   Limite inferior : -2100.00
   Limite superior : 4300.00
   Outliers        : 4551 (13.81%)
   Valor maximo    : 40425.00
   Valor minimo    : 2.00

product_length_cm
   Limite inferior : -12.00
   Limite superior : 68.00
   Outliers        : 1380 (4.19%)
   Valor maximo    : 105.00
   Valor minimo    : 7.00

product_height_cm
   Limite inferior : -11.50
   Limite superior : 40.50
   Outliers        : 1892 (5.74%)
   Valor maximo    : 105.00
   Valor minimo    : 2.00

product_width_cm
   Limite inferior : -7.50
   Limite superior : 52.50
   Outliers        : 912 (2.77%)
   Valor maximo    : 118.00
   Valor minimo    : 6.00


In [ ]:
# Investigando valores suspeitos
print(order_payments[order_payments['payment_installments'] == 0]['payment_type'].value_counts())
print(products[products['product_weight_g'] == 0].shape)
print(order_payments[order_payments['payment_value'] > 10000].shape)

payment_type
credit_card    2
Name: count, dtype: int64
(4, 9)
(1, 5)


In [ ]:
# Removendo pedidos com 0 parcelas no cartao de credito
order_payments = order_payments[~((order_payments['payment_installments'] == 0) & (order_payments['payment_type'] == 'credit_card'))]

In [ ]:
# Removendo produtos com peso zero
products = products[products['product_weight_g'] > 0]

# Saving Cleaned Datasets

In [116]:
# Salvando datasets limpos na pasta processed

customers.to_csv('../data/processed/customers_cleaned.csv', index=False)
geolocation.to_csv('../data/processed/geolocation_cleaned.csv', index=False)
order_items.to_csv('../data/processed/order_items_cleaned.csv', index=False)
order_payments.to_csv('../data/processed/order_payments_cleaned.csv', index=False)
order_reviews.to_csv('../data/processed/order_reviews_cleaned.csv', index=False)
orders.to_csv('../data/processed/orders_cleaned.csv', index=False)
products.to_csv('../data/processed/products_cleaned.csv', index=False)
sellers.to_csv('../data/processed/sellers_cleaned.csv', index=False)
product_category.to_csv('../data/processed/product_category_cleaned.csv', index=False)